In [11]:
import json
from typing import Dict, List, Tuple, Union, Any

import PyPDF2
import pdfplumber

# 1)
pdf_path: str = "C:/Users/6muni/Documents/sample.pdf"


# 2)
def extract_with_pypdf2(pdf_path: str) -> Tuple[List[Dict[str, Union[int, str]]], int]:
    """Извлечение текста с помощью PyPDF2"""
    text_data: List[Dict[str, Union[int, str]]] = []
    with open(pdf_path, "rb") as file:
        pdf_reader: PyPDF2.PdfReader = PyPDF2.PdfReader(file)
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text: str = page.extract_text()
            text_data.append({"page_num": page_num + 1, "text": text})
    return text_data, len(pdf_reader.pages)


# 3)
def extract_with_pdfplumber(pdf_path: str) -> Tuple[List[Dict[str, Union[int, str]]], int]:
    """Извлечение текста с помощью pdfplumber"""
    text_data: List[Dict[str, Union[int, str]]] = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text: str = page.extract_text()
            text_data.append({"page_num": page_num, "text": text or ""})
    return text_data, len(pdf.pages)


# 4)
def compare_extraction(pdf_path: str) -> Tuple[List[Dict[str, Union[int, str]]], List[Dict[str, Union[int, str]]]]:
    """Сравнение качества извлечения"""
    # === PyPDF2 ===
    pypdf2_data: List[Dict[str, Union[int, str]]]
    pypdf2_pages: int
    pypdf2_data, pypdf2_pages = extract_with_pypdf2(pdf_path)

    # === pdfplumber ===
    pdfplumber_data: List[Dict[str, Union[int, str]]]
    pdfplumber_pages: int
    pdfplumber_data, pdfplumber_pages = extract_with_pdfplumber(pdf_path)

    # Сравнение количества страниц
    print(f"\nСравнение:")
    print(
        f"Количество страниц - PyPDF2: {pypdf2_pages}, pdfplumber: {pdfplumber_pages}"
    )

    # Сравнение длины текста на первой странице
    if pypdf2_data and pdfplumber_data:
        pypdf2_len: int = len(pypdf2_data[0]["text"])
        pdfplumber_len: int = len(pdfplumber_data[0]["text"])
        print(
            f"Текст на странице 1 - PyPDF2: {pypdf2_len} символов, pdfplumber: {pdfplumber_len} символов"
        )

    return pypdf2_data, pdfplumber_data


def save_to_json(data: Dict[str, Any], filename: str = "extracted_text.json") -> None:
    """Сохранение в JSON файл"""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"\nРезультат сохранен в {filename}")


# Основной скрипт
try:
    pypdf2_data: List[Dict[str, Union[int, str]]]
    pdfplumber_data: List[Dict[str, Union[int, str]]]
    pypdf2_data, pdfplumber_data = compare_extraction(pdf_path)

    # 6) Сохраняем результат из pdfplumber
    result: Dict[str, Union[str, int, List[Dict[str, Union[int, str]]]]] = {
        "file_file": pdf_path,
        "total_pages": len(pdfplumber_data),
        "pages": pdfplumber_data,
    }

    save_to_json(result)

except FileNotFoundError:
    print(
        f"Файл {pdf_path} не найден. Создайте sample.pdf или укажите путь к существующему PDF файлу."
    )
except Exception as e:
    print(f"Ошибка: {e}")


# TODO:
# 1) Найти любой PDF файл (или скачать sample PDF)
# 2) Загрузить через PyPDF2
# 3) Загрузить через pdfplumber
# 4) Сравнить качество извлечения текста
# 5) Извлечь текст постранично
# 6) Сохранить в структурированном виде:
#    {
#      "file_path": str,
#      "total_pages": int,
#      "pages": [
#        {"page_num": 1, "text": "..."},
#        {"page_num": 2, "text": "..."}
#      ]
#    }


Сравнение:
Количество страниц - PyPDF2: 3, pdfplumber: 3
Текст на странице 1 - PyPDF2: 2977 символов, pdfplumber: 2962 символов

Результат сохранен в extracted_text.json


In [12]:
import sys
from typing import Dict, Union, List

# Добавляем путь к папке src
sys.path.append("../src")

# Импортирование функции
from document_loader import load_pdf

# Тестируем функцию
pdf_path: str = "C:/Users/6muni/Documents/sample.pdf"
result: Dict[str, Union[bool, str, int, List[Dict[str, Union[int, str]]]]] = load_pdf(pdf_path)

if result["success"]:
    print(f"Успешно загружено!")
    print(f"Файл: {result['file_path']}")
    print(f"Страниц: {result['total_pages']}")

    # Показываем первые 3 страницы и часть текста
    pages: List[Dict[str, Union[int, str]]] = result["pages"]  # type: ignore
    for page in pages[:3]:
        text: str = str(page["text"])
        preview: str = text[:100].replace("\n", " ")
        print(f"--- Страница {page['page_num']}: {preview}...")
else:
    print(f"Ошибка: {result['error']}")

Успешно загружено!
Файл: C:\Users\6muni\Documents\sample.pdf
Страниц: 3
--- Страница 1: Sample PDF Created for testing PDFObject This PDF is three pages long. Three long pages. Or three sh...
--- Страница 2: ipsum dolor sit amet, consectetur adipiscing elit. Integer nec odio. Praesent libero. Sed cursus ant...
--- Страница 3: elementum. Morbi in ipsum sit amet pede facilisis laoreet. Donec lacus nunc, viverra nec, blandit ve...
